<a href="https://colab.research.google.com/github/sami01979/fer_pattern/blob/main/fer_pattern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())

!mkdir -p /content/raf
!unzip -q /content/DATASET.zip -d /content/raf
!find /content/raf -maxdepth 3 -type d | sort

2.11.0+cu130 True
/content/raf
/content/raf/DATASET
/content/raf/DATASET/test
/content/raf/DATASET/test/1
/content/raf/DATASET/test/2
/content/raf/DATASET/test/3
/content/raf/DATASET/test/4
/content/raf/DATASET/test/5
/content/raf/DATASET/test/6
/content/raf/DATASET/test/7
/content/raf/DATASET/train
/content/raf/DATASET/train/1
/content/raf/DATASET/train/2
/content/raf/DATASET/train/3
/content/raf/DATASET/train/4
/content/raf/DATASET/train/5
/content/raf/DATASET/train/6
/content/raf/DATASET/train/7


In [2]:
import os
base = "/content/raf/DATASET"
for split in ["train", "test"]:
    counts = {c: len(os.listdir(f"{base}/{split}/{c}")) for c in sorted(os.listdir(f"{base}/{split}"))}
    print(split, counts, "total:", sum(counts.values()))


train {'1': 1290, '2': 281, '3': 717, '4': 4772, '5': 1982, '6': 705, '7': 2524} total: 12271
test {'1': 329, '2': 74, '3': 160, '4': 1185, '5': 478, '6': 162, '7': 680} total: 3068


In [3]:
%%writefile train_fer.py
import math, time, argparse, random, os, csv
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.utils.flop_counter import FlopCounterMode
from torchvision import datasets, transforms, models
from torchvision.models.resnet import BasicBlock
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from tqdm import tqdm


class ECA(nn.Module):
    def __init__(self, c, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(c) + b) / gamma))
        k = t if t % 2 else t + 1
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        y = F.adaptive_avg_pool2d(x, 1).squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * torch.sigmoid(y)

class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(c, c // r), nn.ReLU(inplace=True), nn.Linear(c // r, c))
    def forward(self, x):
        y = self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))
        return x * torch.sigmoid(y)[:, :, None, None]

class CBAM(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.mlp = nn.Sequential(nn.Conv2d(c, c // r, 1, bias=False), nn.ReLU(inplace=True),
                                 nn.Conv2d(c // r, c, 1, bias=False))
        self.spatial = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        a = self.mlp(F.adaptive_avg_pool2d(x, 1))
        m = self.mlp(F.adaptive_max_pool2d(x, 1))
        x = x * torch.sigmoid(a + m)
        s = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)
        return x * torch.sigmoid(self.spatial(s))

ATTN = {"eca": ECA, "se": SE, "cbam": CBAM}

class AttnBasicBlock(BasicBlock):
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.attn(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

def build_model(attn="none", num_classes=7, pretrained=True):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    if attn != "none":
        for layer in [m.layer1, m.layer2, m.layer3, m.layer4]:
            for blk in layer:
                blk.attn = ATTN[attn](blk.bn2.num_features)
                blk.__class__ = AttnBasicBlock
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_loaders(root, bs, workers, size):
    train_tf = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.9, 1.1)),
        transforms.ColorJitter(0.3, 0.3, 0.2),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((size, size)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

    full_train = datasets.ImageFolder(f"{root}/train", train_tf)
    full_eval = datasets.ImageFolder(f"{root}/train", eval_tf)
    test_set = datasets.ImageFolder(f"{root}/test", eval_tf)

    targets = np.array(full_train.targets)

    tr_idx, va_idx = train_test_split(np.arange(len(targets)), test_size=0.1,
                                      stratify=targets, random_state=42)
    counts = np.bincount(targets[tr_idx], minlength=len(full_train.classes))
    weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float)

    kw = dict(num_workers=workers, pin_memory=True)
    return (DataLoader(Subset(full_train, tr_idx), bs, shuffle=True, drop_last=True, **kw),
            DataLoader(Subset(full_eval, va_idx), bs, **kw),
            DataLoader(test_set, bs, **kw),
            full_train.classes, weights)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        ps.append(model(x.to(device)).argmax(1).cpu()); ys.append(y)
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return 100 * (y == p).mean(), 100 * f1_score(y, p, average="macro"), y, p

def fps_of(model, size, device, n=100):
    model = model.to(device).eval()
    x = torch.randn(1, 3, size, size, device=device)
    with torch.no_grad():
        for _ in range(20): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(n): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
    return n / (time.time() - t)

def efficiency(model, size, device):
    params = sum(p.numel() for p in model.parameters()) / 1e6
    cpu_model = model.to("cpu").eval()
    with FlopCounterMode(display=False) as fc:
        cpu_model(torch.randn(1, 3, size, size))
    gmacs = fc.get_total_flops() / 2 / 1e9
    fps_cpu = fps_of(cpu_model, size, torch.device("cpu"), n=30)
    fps_gpu = fps_of(model, size, device) if device.type == "cuda" else float("nan")
    return params, gmacs, fps_cpu, fps_gpu

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--attn", default="eca", choices=["none", "se", "cbam", "eca"])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--size", type=int, default=112)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--out", default="best.pth")
    ap.add_argument("--log", default="results.csv")
    a = ap.parse_args()

    seed_everything(a.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_dl, val_dl, test_dl, classes, cw = get_loaders(a.data, a.bs, a.workers, a.size)
    model = build_model(a.attn, len(classes)).to(device)

    crit = nn.CrossEntropyLoss(weight=cw.to(device), label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=a.epochs)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_f1 = -1.0
    for ep in range(1, a.epochs + 1):
        model.train(); loss_sum = 0.0
        for x, y in tqdm(train_dl, desc=f"Epoch {ep}/{a.epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            loss_sum += loss.item()
        sched.step()
        v_acc, v_f1, _, _ = evaluate(model, val_dl, device)
        print(f"Epoch {ep}: loss {loss_sum/len(train_dl):.4f} | val acc {v_acc:.2f} | val macro-F1 {v_f1:.2f}")
        if v_f1 > best_f1:
            best_f1 = v_f1; torch.save(model.state_dict(), a.out)

    model.load_state_dict(torch.load(a.out, map_location=device))
    t_acc, t_f1, y, p = evaluate(model, test_dl, device)
    cm = confusion_matrix(y, p)
    per_class = 100 * cm.diagonal() / cm.sum(1)
    print(f"\n[{a.attn} | seed {a.seed}] TEST acc {t_acc:.2f} | macro-F1 {t_f1:.2f}")
    print("Per-class acc:", {c: round(float(v), 1) for c, v in zip(classes, per_class)})
    print("Confusion matrix:\n", cm)

    params, gmacs, fps_cpu, fps_gpu = efficiency(model, a.size, device)
    print(f"Params {params:.3f}M | GMACs {gmacs:.3f} | FPS cpu {fps_cpu:.1f} | FPS gpu {fps_gpu:.1f}")

    new = not os.path.exists(a.log)
    with open(a.log, "a", newline="") as f:
        w = csv.writer(f)
        if new: w.writerow(["attn", "seed", "size", "best_val_f1", "test_acc", "test_f1",
                            "params_M", "gmacs", "fps_cpu", "fps_gpu"])
        w.writerow([a.attn, a.seed, a.size, round(best_f1, 2), round(t_acc, 2), round(t_f1, 2),
                    round(params, 4), round(gmacs, 4), round(fps_cpu, 1), round(fps_gpu, 1)])

if __name__ == "__main__":
    main()

Writing train_fer.py


In [4]:
!python train_fer.py --data /content/raf/DATASET --attn eca --epochs 1 --out test.pth --log test.csv

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 212MB/s]
Epoch 1: loss 1.7668 | val acc 58.47 | val macro-F1 52.47

[eca | seed 0] TEST acc 59.16 | macro-F1 51.70
Per-class acc: {'1': 83.3, '2': 37.8, '3': 58.1, '4': 60.0, '5': 54.0, '6': 71.0, '7': 49.4}
Confusion matrix:
 [[274  16  15   4   2   7  11]
 [ 19  28   8   1   5  11   2]
 [ 11   8  93   3   6  12  27]
 [ 62  61 166 711  41  93  51]
 [ 17  17  92  16 258  32  46]
 [ 11   4  22   3   4 115   3]
 [ 82  16 158  12  44  32 336]]
Params 11.180M | GMACs 0.485 | FPS cpu 34.5 | FPS gpu 275.8


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!python train_fer.py --data /content/raf/DATASET --attn none --seed 0 --out none_s0.pth

Epoch 1: loss 1.8077 | val acc 46.82 | val macro-F1 42.07
Epoch 2: loss 1.4975 | val acc 70.20 | val macro-F1 58.77
Epoch 3: loss 1.3990 | val acc 68.24 | val macro-F1 60.76
Epoch 4: loss 1.3292 | val acc 73.86 | val macro-F1 65.94
Epoch 5: loss 1.2893 | val acc 73.78 | val macro-F1 65.66
Epoch 6: loss 1.2282 | val acc 64.66 | val macro-F1 60.43
Epoch 7: loss 1.2101 | val acc 72.56 | val macro-F1 63.52
Epoch 8: loss 1.1674 | val acc 73.86 | val macro-F1 66.87
Epoch 9: loss 1.1298 | val acc 74.92 | val macro-F1 68.30
Epoch 10: loss 1.1377 | val acc 74.10 | val macro-F1 65.32
Epoch 11: loss 1.0867 | val acc 72.31 | val macro-F1 66.34
Epoch 12: loss 1.0790 | val acc 75.57 | val macro-F1 67.80
Epoch 13: loss 1.0362 | val acc 79.97 | val macro-F1 71.16
Epoch 14: loss 1.0155 | val acc 76.14 | val macro-F1 68.37
Epoch 15: loss 1.0050 | val acc 71.82 | val macro-F1 65.80
Epoch 16: loss 0.9758 | val acc 79.32 | val macro-F1 72.30
Epoch 17: loss 0.9517 | val acc 78.34 | val macro-F1 71.51
Epoch 

In [7]:
!for s in 0 1 2; do python train_fer.py --data /content/raf/DATASET --attn eca --seed $s --out eca_s$s.pth; done
!cp results.csv *.pth /content/drive/MyDrive/

Epoch 1: loss 1.7668 | val acc 58.47 | val macro-F1 52.47
Epoch 2: loss 1.4980 | val acc 65.47 | val macro-F1 56.44
Epoch 3: loss 1.3800 | val acc 68.97 | val macro-F1 62.30
Epoch 4: loss 1.3371 | val acc 63.60 | val macro-F1 58.37
Epoch 5: loss 1.2762 | val acc 73.86 | val macro-F1 65.89
Epoch 6: loss 1.2414 | val acc 75.90 | val macro-F1 69.79
Epoch 7: loss 1.1994 | val acc 79.48 | val macro-F1 70.23
Epoch 8: loss 1.1715 | val acc 72.31 | val macro-F1 63.95
Epoch 9: loss 1.1375 | val acc 73.62 | val macro-F1 68.37
Epoch 10: loss 1.1093 | val acc 78.34 | val macro-F1 71.43
Epoch 11: loss 1.0826 | val acc 75.00 | val macro-F1 67.64
Epoch 12: loss 1.0675 | val acc 78.50 | val macro-F1 72.13
Epoch 13: loss 1.0451 | val acc 79.15 | val macro-F1 71.07
Epoch 14: loss 1.0147 | val acc 77.85 | val macro-F1 70.18
Epoch 15: loss 1.0016 | val acc 76.79 | val macro-F1 69.17
Epoch 16: loss 0.9696 | val acc 78.75 | val macro-F1 71.52
Epoch 17: loss 0.9566 | val acc 75.73 | val macro-F1 68.79
Epoch 

In [8]:
!for s in 0 1 2; do python train_fer.py --data /content/raf/DATASET --attn cbam --seed $s --out cbam_s$s.pth; done
!cp results.csv *.pth /content/drive/MyDrive/

Epoch 1: loss 1.8478 | val acc 63.11 | val macro-F1 53.76
Epoch 2: loss 1.5493 | val acc 66.12 | val macro-F1 59.77
Epoch 3: loss 1.4324 | val acc 71.25 | val macro-F1 62.45
Epoch 4: loss 1.3597 | val acc 71.25 | val macro-F1 62.53
Epoch 5: loss 1.3041 | val acc 71.50 | val macro-F1 63.58
Epoch 6: loss 1.2715 | val acc 67.18 | val macro-F1 59.32
Epoch 7: loss 1.2301 | val acc 73.53 | val macro-F1 65.32
Epoch 8: loss 1.1948 | val acc 76.63 | val macro-F1 69.89
Epoch 9: loss 1.1732 | val acc 77.28 | val macro-F1 68.31
Epoch 10: loss 1.1371 | val acc 75.90 | val macro-F1 68.32
Epoch 11: loss 1.1144 | val acc 73.05 | val macro-F1 65.07
Epoch 12: loss 1.0917 | val acc 74.92 | val macro-F1 66.51
Epoch 13: loss 1.0885 | val acc 78.66 | val macro-F1 70.94
Epoch 14: loss 1.0523 | val acc 78.66 | val macro-F1 69.40
Epoch 15: loss 1.0325 | val acc 79.80 | val macro-F1 71.33
Epoch 16: loss 1.0113 | val acc 80.94 | val macro-F1 73.13
Epoch 17: loss 0.9938 | val acc 79.72 | val macro-F1 71.60
Epoch 

In [9]:
!for s in 1 2; do python train_fer.py --data /content/raf/DATASET --attn none --seed $s --out none_s$s.pth; done
!for s in 0 1 2; do python train_fer.py --data /content/raf/DATASET --attn se --seed $s --out se_s$s.pth; done
!cp results.csv *.pth /content/drive/MyDrive/

Epoch 1: loss 1.8055 | val acc 65.39 | val macro-F1 55.65
Epoch 2: loss 1.5038 | val acc 62.05 | val macro-F1 56.60
Epoch 3: loss 1.3913 | val acc 72.72 | val macro-F1 64.92
Epoch 4: loss 1.3306 | val acc 69.79 | val macro-F1 61.86
Epoch 5: loss 1.2782 | val acc 70.11 | val macro-F1 61.79
Epoch 6: loss 1.2402 | val acc 69.71 | val macro-F1 63.86
Epoch 7: loss 1.2072 | val acc 68.32 | val macro-F1 62.26
Epoch 8: loss 1.1714 | val acc 73.37 | val macro-F1 64.99
Epoch 9: loss 1.1321 | val acc 75.08 | val macro-F1 65.66
Epoch 10: loss 1.1109 | val acc 73.29 | val macro-F1 65.09
Epoch 11: loss 1.0910 | val acc 77.12 | val macro-F1 69.69
Epoch 12: loss 1.0553 | val acc 70.77 | val macro-F1 63.47
Epoch 13: loss 1.0244 | val acc 78.34 | val macro-F1 70.63
Epoch 14: loss 1.0145 | val acc 79.72 | val macro-F1 71.90
Epoch 15: loss 0.9945 | val acc 76.47 | val macro-F1 67.99
Epoch 16: loss 0.9716 | val acc 77.28 | val macro-F1 69.83
Epoch 17: loss 0.9620 | val acc 80.37 | val macro-F1 72.57
Epoch 

In [ ]:
import pandas as pd
df = pd.read_csv("results.csv", keep_default_na=False)
g = df.groupby("attn").agg(
    n=("seed", "count"),
    acc_mean=("test_acc", "mean"), acc_std=("test_acc", "std"),
    f1_mean=("test_f1", "mean"), f1_std=("test_f1", "std"),
    params_M=("params_M", "first"), gmacs=("gmacs", "first"),
    fps_cpu=("fps_cpu", "mean"), fps_gpu=("fps_gpu", "mean")).round(2)
print(g)